# RF-DETR Temporal vs TransVOD++ Swin-B: speed benchmark

Notebook предназначен только для измерения скорости двух моделей на 300 ключевых кадрах компактного датасета. Обе модели запускаются в одном Colab GPU-runtime с `batch_size=1` и `FP32`. Время чтения файлов, преобразования изображений, передачи тензоров на GPU, постобработки и фильтрации классов `car`/`bus` не входит в замер.

Перед запуском выберите **Runtime -> Change runtime type -> T4 GPU**, затем выполняйте ячейки сверху вниз.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

def run_with_log(command, *, cwd, env, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Running:', ' '.join(map(str, command)))
    with log_path.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            command, cwd=cwd, env=env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
        )
        recent = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            recent.append(line.rstrip())
            recent = recent[-80:]
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f'Command failed with exit code {return_code}. Log: {log_path}\n'
            + '\n'.join(recent)
        )

print('Python:', sys.version)
print('Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=True)

## 1. Подключение Google Drive и проверка файлов

Ожидается папка `MyDrive/rf-detr-colab` с двумя архивами проектов, компактным датасетом и двумя checkpoint-файлами.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/rf-detr-colab')
RF_ARCHIVE = DRIVE_ROOT / 'rf-detr-temporal.zip'
TRANSVOD_ARCHIVE = DRIVE_ROOT / 'TransVOD_plusplus_swinb_car_bus_eval.zip'
DATASET_ARCHIVE = DRIVE_ROOT / 'temporal_dataset_transvod_selected.zip'
RF_CHECKPOINT_SOURCE = DRIVE_ROOT / 'checkpoint_29.ckpt'
TRANSVOD_CHECKPOINT_SOURCE = DRIVE_ROOT / 'transvodpp_swinb_imagenet_vid.pth'

RF_ROOT = Path('/content/rf-detr-temporal')
TRANSVOD_ROOT = Path('/content/TransVOD_plusplus_swinb_car_bus_eval')
DATASET_ROOT = Path('/content/temporal_dataset_transvod_selected')

required = [RF_ARCHIVE, TRANSVOD_ARCHIVE, DATASET_ARCHIVE, RF_CHECKPOINT_SOURCE, TRANSVOD_CHECKPOINT_SOURCE]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Files missing from Google Drive:\n' + '\n'.join(missing))
print('All source files found in:', DRIVE_ROOT)

In [ ]:
def extract_archive(archive: Path, expected_root: Path, replace: bool = False) -> None:
    if replace and expected_root.exists():
        print('Removing stale project:', expected_root)
        shutil.rmtree(expected_root)
    if expected_root.exists():
        print('Reusing:', expected_root)
        return
    print('Extracting:', archive.name)
    shutil.unpack_archive(str(archive), '/content')
    if not expected_root.exists():
        raise FileNotFoundError(f'{archive.name} did not create {expected_root}')

extract_archive(RF_ARCHIVE, RF_ROOT, replace=True)
extract_archive(TRANSVOD_ARCHIVE, TRANSVOD_ROOT, replace=True)
extract_archive(DATASET_ARCHIVE, DATASET_ROOT)

rf_checkpoint = RF_ROOT / 'result/colab_checkpoint/checkpoint_29.ckpt'
rf_checkpoint.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(RF_CHECKPOINT_SOURCE, rf_checkpoint)

transvod_checkpoint = TRANSVOD_ROOT / 'checkpoints/transvodpp_swinb_imagenet_vid.pth'
transvod_checkpoint.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(TRANSVOD_CHECKPOINT_SOURCE, transvod_checkpoint)

assert (DATASET_ROOT / 'annotations/instances_train2017.json').is_file()
assert (RF_ROOT / 'tools/benchmark_inference.py').is_file()
assert (TRANSVOD_ROOT / 'benchmark.py').is_file()
print('Projects, compact dataset and checkpoints are ready.')

## 2. RF-DETR Temporal: установка и замер

Используется checkpoint с индексом 29, три опорных кадра, разрешение `640x640`, 10 прогревочных и 300 измеряемых проходов по разделу `train`.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
# Colab may preinstall torchao 0.10.0. Current PEFT detects it but only
# supports torchao > 0.16.0. TorchAO is optional and unused by this LoRA model.
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
    check=False,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', '.[train,visual]'],
    cwd=RF_ROOT / 'external/rf-detr',
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install',
        'pyDeprecate>=0.6,<0.8', 'tensorboard', 'opencv-python-headless',
    ],
    check=True,
)

In [ ]:
rf_source = str(RF_ROOT / 'external/rf-detr/src')
if rf_source not in sys.path:
    sys.path.insert(0, rf_source)

import torch
import rfdetr

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a GPU runtime in Colab.')
print('RF-DETR import: OK')
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
rf_output = RF_ROOT / 'reports/inference_speed/temporal_car_bus_train_300.json'
rf_log = DRIVE_ROOT / 'results/speed/rf_detr_temporal_benchmark.log'
rf_output.parent.mkdir(parents=True, exist_ok=True)
rf_env = os.environ.copy()
rf_env.update({
    'DEVICE': 'cuda:0',
    'PYTHON_BIN': sys.executable,
    'SPLIT': 'train',
})
run_with_log([
    'bash', 'commands/benchmark_temporal_car_bus_300.sh',
    str(DATASET_ROOT), str(rf_checkpoint), str(rf_output),
], cwd=RF_ROOT, env=rf_env, log_path=rf_log)
print('RF-DETR benchmark saved to:', rf_output)

## 3. TransVOD++ Swin-B: установка и замер

TransVOD++ устанавливается в отдельную Conda-среду Python 3.8. Используются штатные 14 опорных кадров модели, 10 прогревочных и 300 измеряемых проходов по тому же разделу `train`. Размер входа остается оригинальным для TransVOD++: короткая сторона 600 пикселей, длинная - не более 1000 пикселей.

In [ ]:
import urllib.request

conda_root = Path('/content/miniconda')
conda_exe = conda_root / 'bin/conda'
if not conda_exe.exists():
    installer = Path('/content/miniconda.sh')
    urllib.request.urlretrieve(
        'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh',
        installer,
    )
    subprocess.run(['bash', str(installer), '-b', '-p', str(conda_root)], check=True)
else:
    print('Reusing existing Miniconda installation.')

setup_env = os.environ.copy()
setup_env['PATH'] = f'{conda_root}/bin:' + setup_env['PATH']
setup_env['ENV_NAME'] = 'transvodpp-swinb'
transvod_setup_log = DRIVE_ROOT / 'results/speed/transvodpp_setup.log'
run_with_log(
    ['bash', 'setup_ubuntu.sh'],
    cwd=TRANSVOD_ROOT,
    env=setup_env,
    log_path=transvod_setup_log,
)

In [ ]:
verification = (
    "import torch; "
    "from models.ops.modules import MSDeformAttn; "
    "print('PyTorch:', torch.__version__); "
    "print('CUDA runtime:', torch.version.cuda); "
    "print('CUDA available:', torch.cuda.is_available()); "
    "print('GPU:', torch.cuda.get_device_name(0))"
)
subprocess.run([
    str(conda_exe), 'run', '--no-capture-output', '-n', 'transvodpp-swinb',
    'python', '-c', verification,
], cwd=TRANSVOD_ROOT, check=True)

In [ ]:
transvod_output = TRANSVOD_ROOT / 'outputs/inference_speed/transvodpp_swinb_car_bus_train_300.json'
transvod_output.parent.mkdir(parents=True, exist_ok=True)
benchmark_command = (
    f'cd {TRANSVOD_ROOT} && '
    f'BENCHMARK_SPLIT=train bash scripts/benchmark_swinb_car_bus_14refs.sh '
    f'{DATASET_ROOT} {transvod_checkpoint} {transvod_output}'
)
subprocess.run([
    str(conda_exe), 'run', '--no-capture-output', '-n', 'transvodpp-swinb',
    'bash', '-lc', benchmark_command,
], check=True)
print('TransVOD++ benchmark saved to:', transvod_output)

## 4. Сводная таблица и сохранение результатов

`FPS` означает число предсказаний для ключевых кадров в секунду. Фильтрация `car`/`bus` выполняется после остановки таймера и не влияет на FPS. Сравнение отражает реальные конфигурации моделей: RF-DETR Temporal обрабатывает 4 кадра на одно предсказание, TransVOD++ - 15 кадров.

In [ ]:
import json
import pandas as pd

rf_output = RF_ROOT / 'reports/inference_speed/temporal_car_bus_train_300.json'
transvod_output = TRANSVOD_ROOT / 'outputs/inference_speed/transvodpp_swinb_car_bus_train_300.json'
missing_results = [str(path) for path in (rf_output, transvod_output) if not path.is_file()]
if missing_results:
    raise FileNotFoundError(
        'Run the benchmark cells that produce these files before comparison:\n'
        + '\n'.join(missing_results)
    )

rf_result = json.loads(rf_output.read_text(encoding='utf-8'))
transvod_result = json.loads(transvod_output.read_text(encoding='utf-8'))
comparison = pd.DataFrame([
    {
        'model': 'RF-DETR Temporal',
        'split': rf_result['split'],
        'classes': 'car, bus',
        'reference_frames': rf_result['input_frames_per_key_image'] - 1,
        'input_policy': '640x640',
        'measured_key_frames': rf_result['measured_key_images'],
        'mean_latency_ms': rf_result['latency_mean_ms'],
        'median_latency_ms': rf_result['latency_median_ms'],
        'p95_latency_ms': rf_result['latency_p95_ms'],
        'fps': rf_result['fps'],
        'peak_vram_gb': rf_result['peak_vram_gb'],
    },
    {
        'model': 'TransVOD++ Swin-B',
        'split': transvod_result['split'],
        'classes': 'car, bus',
        'reference_frames': transvod_result['architecture']['num_ref_frames'],
        'input_policy': 'short=600, max=1000',
        'measured_key_frames': transvod_result['measured_key_images'],
        'mean_latency_ms': transvod_result['latency_mean_ms'],
        'median_latency_ms': transvod_result['latency_median_ms'],
        'p95_latency_ms': transvod_result['latency_p95_ms'],
        'fps': transvod_result['fps'],
        'peak_vram_gb': transvod_result['peak_vram_gb'],
    },
])
display(comparison.style.format({
    'mean_latency_ms': '{:.3f}',
    'median_latency_ms': '{:.3f}',
    'p95_latency_ms': '{:.3f}',
    'fps': '{:.3f}',
    'peak_vram_gb': '{:.3f}',
}))

In [ ]:
results_drive = DRIVE_ROOT / 'results/speed'
results_drive.mkdir(parents=True, exist_ok=True)
comparison_path = results_drive / 'rf_detr_temporal_vs_transvodpp_swinb_speed.csv'
comparison.to_csv(comparison_path, index=False)
shutil.copy2(rf_output, results_drive / rf_output.name)
shutil.copy2(transvod_output, results_drive / transvod_output.name)
print('Results saved to:', results_drive)
print('Comparison:', comparison_path)

## Методика интерпретации

Основной показатель - `fps`, рассчитанный как `1000 / mean_latency_ms`. Для оценки разброса дополнительно сохраняются медианная задержка и 95-й процентиль. Поскольку модели используют разное число опорных кадров и разные штатные правила масштабирования, таблица сравнивает скорость их фактических конфигураций, а не только скорость одинакового backbone-входа.